# core.dom.summary.task

> Identify task tables in saved object-summary documents without generating summaries.

A `task` is a semantic subtype of `table` in the existing JSON format. It keeps
the same content, rows, merged groups, source locations, Pandoc subtree and
summaries; only the object's `type` changes. No Python table class is required.

In [ ]:
# | default_exp core.dom.summary.task

## Three independent rules

A table qualifies if **any** of these conditions holds:

1. An enclosing section title contains `作业模板`, `模板作业`, `作业示例`, or
   `示例作业`. All nested subsections inherit this context.
2. A cell in the **first physical row** must equal `作业`, apart from surrounding
   spaces or tabs. This includes HTML `<td>` cells and rows nested in merged
   row groups. Longer labels such as `作业名` or `作业内容` do not match. Later rows never trigger
   this rule, even if a source `<thead>` incorrectly wraps the entire table.
3. The immediately preceding title contains `子作业参数表` or `基本作业流程`.
   A nonempty attached caption takes precedence; otherwise use the preceding
   paragraph or heading. A section heading is adjacent only to its first body
   block. Blank lines do not break adjacency; other content does.

Readable text ignores formatting and link targets. Missing row metadata is parsed
from the saved table content using the existing table utilities, solely for
detection. The source Markdown is not needed or opened.

In [ ]:
# | export
from copy import deepcopy
from html.parser import HTMLParser
import re

from markdown_it import MarkdownIt
from ribosome.core.dom.summary.object_utils import _inline_text, _table_row_records


SECTION_TASK_TERMS = ('作业模板', '模板作业', '作业示例', '示例作业')
TABLE_TASK_TERMS = ('子作业参数表', '基本作业流程')


class _HTMLText(HTMLParser):
    def __init__(self, source):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self.feed(source)
        self.close()

    def handle_starttag(self, tag, attrs):
        if tag in {'br', 'p', 'div', 'li', 'tr', 'td', 'th'}:
            self.parts.append(' ')

    def handle_endtag(self, tag):
        if tag in {'p', 'div', 'li', 'tr', 'td', 'th'}:
            self.parts.append(' ')

    def handle_data(self, data):
        self.parts.append(data)


def _source_text(source):
    """Read visible HTML/Markdown text, retaining adjacent inline text."""
    parts = []
    for token in MarkdownIt('commonmark', {'html': True}).parse(source or ''):
        if token.type == 'inline':
            # Strip inline HTML while preserving its text and decoded entities.
            parts.append(''.join(_HTMLText(_inline_text(token.children or [])).parts))
        elif token.type == 'html_block':
            parts.append(''.join(_HTMLText(token.content).parts))
    return re.sub(r'\s+', ' ', ' '.join(parts)).strip()


def _node_text(node):
    """Read Pandoc visible text without attributes, link targets or summaries."""
    if isinstance(node, str):
        return node
    if isinstance(node, list):
        return ''.join(_node_text(child) for child in node)
    if not isinstance(node, dict):
        return ''
    kind, content = node.get('t'), node.get('c', [])
    if kind in {'Space', 'SoftBreak', 'LineBreak'}:
        return ' '
    if kind in {'Link', 'Image'}:
        return _node_text(content[-2])
    if kind in {'Header', 'Span', 'Code', 'Math', 'Cite', 'Quoted'}:
        return _node_text(content[-1])
    if kind in {'RawBlock', 'RawInline'}:
        return _source_text(content[-1]) if content[0] == 'html' else ''
    if kind == 'Note':
        return ''
    return _node_text(content)


def _title_text(node):
    if isinstance(node, dict) and node.get('t') in {'Para', 'Plain', 'Header'}:
        return re.sub(r'\s+', ' ', _node_text(node)).strip()
    return ''


def _first_row_cells(table):
    def rows(children):
        for child in children:
            if child.get('type') == 'table_row':
                yield child
            elif child.get('type') == 'table_row_group':
                yield from rows(child.get('children', []))

    physical_rows = list(rows(table.get('children', [])))
    if physical_rows:
        first = min(enumerate(physical_rows), key=lambda pair: pair[1].get('row_index', pair[0] + 1))[1]
        if first.get('row_index', 1) == 1 and first.get('cells'):
            return first['cells']
    if table.get('content'):
        scratch = dict(table, id=table.get('id', 'table'), source_file=table.get('source_file', ''),
                       caption=table.get('caption', ''))
        children, _ = _table_row_records(scratch, table.get('line_start', 1))
        return next(rows(children), {}).get('cells', [])
    return []


def task_match_reasons(table: dict, *, section_titles=(), preceding_title: str = '') -> tuple[str, ...]:
    """Return independent task rules matched by a table, without changing it.

    The first physical row counts as the header even when HTML uses td cells.
    Its visible cell text must equal 作业 after trimming spaces and tabs.
    A nonempty attached caption takes precedence over the preceding title.
    """
    if table.get('type') not in {'table', 'task'}:
        return ()
    reasons = []
    if any(term in title for title in section_titles for term in SECTION_TASK_TERMS):
        reasons.append('section_title')
    if any((cell['text'] if isinstance(cell.get('text'), str)
            else _source_text(cell.get('content', ''))).strip(' \t') == '作业'
           for cell in _first_row_cells(table)):
        reasons.append('header_row')
    title = _source_text(table.get('caption', '')) or preceding_title
    if any(term in title for term in TABLE_TASK_TERMS):
        reasons.append('preceding_title')
    return tuple(reasons)

In [ ]:
# | export
def _record_key(record, document):
    if record.get('id') is None:
        raise ValueError('A table/task object must have an id')
    return str(record.get('source_file') or document), str(record['id'])


def task_object_records(tree: dict) -> list[dict]:
    """Report unique matching tables and existing tasks in source order.

    Diagnostic records contain identity, original type, section_path, source
    lines, preceding_title and reasons. No diagnostic fields enter the tree.
    Section context is inherited by all descendants; titles apply only to the
    next nonblank block. Semantic children are visited independently of their
    raw Pandoc subtrees, and mirrored records are deduplicated by identity.
    """
    if not isinstance(tree, dict):
        raise TypeError('Expected a document dictionary')
    document = tree.get('file_path', '')
    records = {}

    def record(obj, sections, title):
        key = _record_key(obj, document)
        reasons = task_match_reasons(obj, section_titles=sections, preceding_title=title)
        if key not in records:
            records[key] = dict(id=obj['id'], source_file=key[0], type=obj['type'],
                                section_path=sections, line_start=obj.get('line_start'),
                                line_end=obj.get('line_end'), preceding_title=title,
                                reasons=reasons)
        else:
            saved = records[key]
            if not saved['reasons'] and reasons:
                saved.update(section_path=sections, preceding_title=title)
            saved['reasons'] = tuple(dict.fromkeys((*saved['reasons'], *reasons)))
            if obj['type'] == 'task':
                saved['type'] = 'task'

    def sequence(values, sections, initial_title=''):
        title = initial_title
        for value in values:
            # Blank lines normally disappear during Pandoc parsing. Also accept
            # explicit empty paragraphs/Null blocks without breaking adjacency.
            if value is None or (isinstance(value, dict) and (
                value.get('t') == 'Null' or
                value.get('t') in {'Para', 'Plain'} and not _title_text(value)
            )):
                continue
            visit(value, sections, title)
            title = _title_text(value)

    def visit(value, sections=(), title=''):
        if isinstance(value, list):
            sequence(value, sections, title)
            return
        if not isinstance(value, dict):
            return
        if value.get('t') == 'Section':
            header, body = value['c']
            heading = _title_text(header)
            current = (*sections, heading)
            # Body Content and structural subsections are separate in this DOM.
            if isinstance(body, dict) and body.get('t') == 'Content':
                sequence(body.get('c', []), current, heading)
            else:
                visit(body, current, heading)
            sequence(value.get('subnodes', []), current)
            return
        if isinstance(value.get('object'), dict):
            visit(value['object'], sections, title)
            sequence(value.get('subnodes', []), sections)
            return
        if value.get('type') in {'table', 'task'}:
            record(value, sections, title)
        for key in ('blocks', 'c', 'children', 'subnodes', 'objects'):
            if key in value:
                visit(value[key], sections)

    visit(tree)
    return [record for record in records.values() if record['reasons'] or record['type'] == 'task']


def classify_task_objects(tree: dict) -> dict:
    """Copy a saved object-summary DOM, changing only matching table types.

    All table data and summaries remain intact. Matching records are upgraded
    everywhere they are mirrored in the JSON; existing tasks are never demoted.
    This function does not read source documents, write files or call a model.
    """
    document = tree.get('file_path', '') if isinstance(tree, dict) else ''
    selected = {(record['source_file'], str(record['id'])) for record in task_object_records(tree)}
    result = deepcopy(tree)

    def apply(value):
        if isinstance(value, dict):
            if value.get('type') == 'table' and _record_key(value, document) in selected:
                value['type'] = 'task'
            for child in value.values():
                apply(child)
        elif isinstance(value, list):
            for child in value:
                apply(child)

    apply(result)
    return result

## Classification and inspection

`classify_task_objects(tree)` returns an independent copy. Mirrored object records
are changed consistently by source file and object ID. Existing tasks remain tasks,
and table rows, row groups, cells, lists and figures keep their types.

`task_object_records(tree)` returns unique matching tables and existing tasks with
section paths, source lines and matching reasons. These diagnostics are kept
outside the document JSON. The application notebook uses them for counts and
previews. Both functions are deterministic and make no model requests.

In [ ]:
# | hide
example = {
    'file_path': 'manual.md',
    'summary': 'Existing document summary',
    'blocks': [{
        't': 'Div', 'c': [['', [], []], []], 's': 'Existing table summary',
        'object': {'id': 1, 'type': 'table', 'source_file': 'manual.md',
                   'content': '| 作业 | 说明 |\n| --- | --- |\n| NOP | 开始 |\n',
                   'caption': '', 'summary': 'Existing table summary'},
    }],
}
classified = classify_task_objects(example)
assert example['blocks'][0]['object']['type'] == 'table'
assert classified['blocks'][0]['object']['type'] == 'task'
assert classified['summary'] == example['summary']
assert classify_task_objects(classified) == classified
assert task_object_records(example)[0]['reasons'] == ('header_row',)